5번 노트북에서

RRF 하이퍼파라미터 튜닝

A안 : Dense 1.0 : Sparse 1.5 (rrf_k=60) <키워드 중심> <br>
B안 : Dense 1.5 : Sparse 1.0 (rrf_k=60) <의미 중심>   <br>
C안 : Dense 1.0 : Sparse 1.0 (rrf_k=40) <RRF 상수변경><br>

In [23]:
import os
import re
import json
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from tqdm import tqdm
from sentence_transformers import SentenceTransformer, CrossEncoder
from dotenv import load_dotenv
from openai import OpenAI

from kiwipiepy import Kiwi
from rank_bm25 import BM25Okapi

## Data & API Key Load & GPU Set

In [24]:
DATA_DIR = Path("../data")
documents_path = DATA_DIR / "documents.jsonl"
eval_path = DATA_DIR / "eval.jsonl"

documents = []
with open(documents_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            documents.append(json.loads(line))

eval_data = []
with open(eval_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            eval_data.append(json.loads(line))

print(f"총 색인 문서 수: {len(documents)}")
print(f"평가 데이터 수: {len(eval_data)}")


# Device Set
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


# API Key Load
load_dotenv(dotenv_path="../.env")

SOLAR_API_KEY = os.getenv("SOLAR_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Solar API Key Load
if not SOLAR_API_KEY:
    print("SOLAR_API_KEY 를 찾을 수 없습니다.")
else:
    print("Solar API Key 가 성공적으로 로드되었습니다.")

# Open API Key Load
if not OPENAI_API_KEY:
    print("OPENAI_API_KEY 를 찾을 수 없습니다.")
else:
    print("OPENAI_API_KEY 가 성공적으로 로드되었습니다.")

총 색인 문서 수: 4272
평가 데이터 수: 220
Using device: cuda
Solar API Key 가 성공적으로 로드되었습니다.
OPENAI_API_KEY 가 성공적으로 로드되었습니다.


## Model

In [25]:
# 1. 임베딩 모델 로드 (Bi-Encoder)
embedding_model_name = "BAAI/bge-m3"
embedding_model = SentenceTransformer(
    embedding_model_name,
    device = device,
    model_kwargs={"use_safetensors": True}
)
print(f"Embedding Model Loaded: {embedding_model}")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding Model Loaded: SentenceTransformer(
  (0): Transformer({'max_seq_length': 8192, 'do_lower_case': False, 'architecture': 'XLMRobertaModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': True, 'pooling_mode_mean_tokens': False, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [26]:
# 2. 리랭커 모델 로드 (Cross-Encoder 역할)
reranker_name = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(reranker_name, device=device)
print(f"Reranker Loaded: {reranker_name}")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Reranker Loaded: BAAI/bge-reranker-v2-m3


## Metadata Split / 전체 문서 임베딩 생성

In [27]:
# 색인용 텍스트 및 메타데이터 분리
texts = [doc["content"] for doc in documents]
metadatas = [
    {
        "docid": doc["docid"],
        "src": doc["src"]
    }
    for doc in documents
]

In [28]:
# 전체 문서 임베딩 생성
# 실제 서비스에서는 이 결과를 Vector DB(FAISS[재현님이 말씀하신 부분], Chroma 등)에 저장
doc_embeddings = embedding_model.encode(
    texts,
    batch_size=32,
    normalize_embeddings=True,      # 코사인 유사도 계산을 위해 정규화
    convert_to_numpy=True,
    show_progress_bar=True
)
print(f"Doc Embeddings Shape: {doc_embeddings.shape}")

Batches:   0%|          | 0/134 [00:00<?, ?it/s]

Doc Embeddings Shape: (4272, 1024)


## Solar(추후 사용시 Code 추가 필요)  & Open AI API Call

In [29]:
# OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

def analyze_and_rewrite_query(msg_list, model="gpt-4o-mini"):
    '''
    멀티턴 대화 기록(msg_list)을 받아 과학 질문 여부를 판별하고,
    단일 질의(Standalone Query) 로 재작성 해가지고 반환하는 Router Funtion
    '''

    # 1. System Prompt : 에이전트의 역할과 규칙을 정의하고, JSON 출력 형식을 강제
    system_prompt = """
    당신은 RAG 시스템의 입력 질의를 분석하는 지능형 라우터입니다.
    사용자의 대화 기록(Multi-turn)을 분석하여 다음 두 가지 작업을 수행하세요.

    Task 1. Intent Classification (의도 분류)
    - 대화의 마지막 메시지가 '지식, 정보, 이유, 장단점'을 묻는 질문인지, 아니면 단순한 '일상 대화(인사, 감정 표현, 칭찬, 잡담)'인지 극단적으로 분류하세요.
    - [절대 규칙]: 질문에 "왜?", "어떻게?", "뭐가 있어?", "어떤 형태?" 같은 의문사가 포함되어 있거나, 무언가(여행, 심리, 사회 현상 등)에 대한 '설명'을 요구한다면 무조건 is_science를 true로 설정하세요. (연애, 심리, 사회학 모두 true입니다.)
    - [절대 규칙]: 오직 "안녕", "고마워", "힘들다", "신난다", "너 똑똑하다" 같이 지식을 요구하지 않는 순수한 감정 표현이나 인사말만 false로 설정하세요.

    Task 2. Standalone Query Rewriting (질의 재작성)
    - is_science가 true인 경우: 이전 대화의 문맥(대명사, 생략된 주어 등)을 파악하여, 검색 엔진(Vector DB)에 단독으로 입력해도 완벽히 의미가 통하는 '하나의 명확한 질문'으로 재작성하세요.
    - is_science가 false인 경우: 빈 문자열("")을 반환하세요.

    [출력 형식]
    반드시 아래의 순수 JSON 포맷으로만 응답하세요. 다른 설명은 절대 추가하지 마세요.
    {
        "is_science": boolean,
        "standalone_query": "string"
    }
    """

    # 2. User Prompt : 실제 대화 기록을 보기 좋게 텍스트로 변환하여 주입
    formatted_history = "\n".join([f"[{turn['role']}] {turn['content']}" for turn in msg_list])
    user_prompt = f"[대화 기록]\n{formatted_history}\n\n위 대화 기록을 분석하여 JSON 형식으로 결과를 출력해 주세요."

    try:
        # 응답 형식을 JSON 으로 강제 (response_format)
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.0,
            response_format={"type": "json_object"}
        )

        # JSON 문자열 -> 딕셔너리로 변환 후 반환
        result_str = response.choices[0].message.content.strip()
        return json.loads(result_str)
    
    except Exception as e:
        print(f"Router API Error: {e}")
        # API Error 반환 시 기본값(Rule-base Fallback 용도) 반환
        return {"is_science": False, "standalone_query": ""}

## Hybrid Search (Dense + Sparse) & RRF

In [30]:
# 1. 한국어 형태소 분석기 초기화
kiwi = Kiwi()

def tokenize(text):
    '''
    텍스트에서 명사(N)와 동사/형용사(V) 등 의미 있는 품사만 추출하여 토큰화합니다.
    (조사나 어미 등 불용어 처리에 효과적)
    '''
    return [token.form for token in kiwi.tokenize(text) if token.tag.startswith('N') or token.tag.startswith('V')]

# 2. 문서 인덱싱 (BM25)
print("BM25 형태소 분석 및 인덱싱 시작")
tokenized_corpus = [tokenize(doc["content"]) for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
print("BM25 인덱싱 완료")

# 빠른 문서를 조회하기 위해서 딕셔너리 매핑 O(1)
doc_lookup = {meta["docid"]: {"content": text, "src": meta["src"]} for meta, text in zip(metadatas, texts)}

BM25 형태소 분석 및 인덱싱 시작
BM25 인덱싱 완료


In [31]:
def hybrid_search(query, k=30, rrf_k=60, dense_weight=1.5, sparse_weight=1.0):
    '''
    Dense(BGE-m3) 랑 Sparse(BM25) 를 동시 수행하고 RRF 로 융합해서 Top-k 문서를 반환
    [B안] Dense(벡터) 가중치를 1.5로 높이고, Sparse(키워드)는 1.0으로 설정합니다.
    '''
    if not query or query == "[]":
        return []
    
    # ------------------------------------
    # 1. Dense Search (Vector)
    # ------------------------------------
    query_embedding = embedding_model.encode([query], normalize_embeddings=True, convert_to_numpy=True)
    dense_scores = np.dot(doc_embeddings, query_embedding[0])
    dense_top_indices = np.argsort(dense_scores)[::-1][:k]

    dense_results = []
    for rank, idx in enumerate(dense_top_indices):
        dense_results.append({"docid": metadatas[idx]["docid"], "dense_rank": rank + 1})
    
    # ------------------------------------
    # 2. Sparse Search (BM25)
    # ------------------------------------
    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_top_indices = np.argsort(bm25_scores)[::-1][:k]

    bm25_results = []
    for rank, idx in enumerate(bm25_top_indices):
        bm25_results.append({"docid": metadatas[idx]["docid"], "bm25_rank": rank + 1})

    # ------------------------------------
    # 3. RRF (가중치 융합!)
    # ------------------------------------
    rrf_scores = {}

    for item in dense_results:
        docid = item["docid"]
        # 🌟 Dense 가중치(1.5) 곱하기
        rrf_scores[docid] = rrf_scores.get(docid, 0.0) + (dense_weight / (rrf_k + item["dense_rank"]))

    for item in bm25_results:
        docid = item["docid"]
        # 🌟 Sparse 가중치(1.0) 곱하기
        rrf_scores[docid] = rrf_scores.get(docid, 0.0) + (sparse_weight / (rrf_k + item["bm25_rank"]))

    # RRF 점수 기준으로 내림차순 정렬 후 Top-k 추출
    sorted_rrf = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)[:k]

    # ------------------------------------
    # 4. 최종 결과 포맷팅
    # ------------------------------------
    final_results = []
    for docid, score in sorted_rrf:
        doc_info = doc_lookup[docid]
        final_results.append({
            "docid": docid,
            "content": doc_info["content"],
            "src": doc_info["src"],
            "rrf_score": score
        })

    return final_results


## Reranking & Answer Generation

In [32]:
# ---------------------------------------------------------
# 1. Reranker 함수 (Cross-Encoder 활용)
# ---------------------------------------------------------
def rerank_hybrid_results(query, hybrid_results, top_k=3):
    '''
    하이브리드 검색으로 얻은 결과(hybrid_results)를 리랭커 모델로 정밀 평가 -> 최종 Top-k 문서를 반환
    '''
    if not hybrid_results:
        return []
    
    # 리랭커 입력 포맷 생성: [[질문, 문서1], [질문, 문서2] ...]
    pairs = [[query, item["content"]] for item in hybrid_results]

    # Cross-Encoder 예측 (show_progress_bar=False 로 깔끔하게 출력)
    rerank_scores = reranker.predict(pairs, batch_size=16, show_progress_bar=False)

    # 기존 결과에 Rerank_score 추가
    for item, score in zip(hybrid_results, rerank_scores):
        item["rerank_score"] = float(score)

    # rerank_score 기준으로 내림차순 정렬
    reranked_results = sorted(hybrid_results, key=lambda x: x["rerank_score"], reverse=True)

    return reranked_results[:top_k]

In [33]:
# ---------------------------------------------------------
# 1-5. JSON 직렬화 에러를 유발하는 Null 바이트 및 눈에 보이지 않는 제어 문자를 지워줘야함.
# ---------------------------------------------------------
def clean_text_for_json(text):
    if not text:
        return ""
    # \n, \t 같은 필수 줄바꿈, 띄어쓰기는 놔두고 에러를 일으키는 찌꺼기만 날리기
    return re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]', '', text)

In [34]:
# ---------------------------------------------------------
# 2. Generator 함수 (OpenAI LLM 활용)
# ---------------------------------------------------------
def generate_final_answer(query, retrieved_docs, model="gpt-4o-mini"):
    if not retrieved_docs:
        return "질문에 답할 수 있는 관련 문서를 찾지 못했어요."

    # 컨텍스트 조립할 때 '정제 함수'를 통과시킵니다!
    context_text = ""
    for i, doc in enumerate(retrieved_docs):
        clean_content = clean_text_for_json(doc['content'])
        context_text += f"[문서 {i+1}]\n{clean_content}\n\n"

    system_prompt = """
    당신은 과학 상식 대회를 위한 똑똑하고 친절한 AI 조수입니다.
    반드시 [제공된 참고 문서]만을 바탕으로 답변하세요.
    
    [답변 규칙]
    1. 대화 스타일: 딱딱한 설명문이 아닌, 친절하게 대화하듯 '구어체(해요체 등 자연스러운 대화 형식)'를 사용하세요.
    2. 가독성: 너무 길지 않게 핵심만 명확하게 짚어서 설명해 주세요.

    [자가 검증 (Self-Correction) - 절대 규칙]
    - 내가 작성한 답변에 [제공된 참고 문서]에 없는 나의 외부 지식이 섞여 있는가?
    - 외부 지식이 섞여 있다면 즉시 삭제하고 오직 문서 내용으로만 재구성하세요.
    """

    # 질문에도 혹시 모를 찌꺼기가 있을 수 있으니 정제
    clean_query = clean_text_for_json(query)

    user_prompt = f"""
    [제공된 참고 문서]
    {context_text}
    
    [사용자 질문]
    {clean_query}
    """

    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            temperature=0.1
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Generator API Error: {e}")
        return "답변 생성 중 오류가 발생했어요."

## HyDE 

In [35]:
# ---------------------------------------------------------
# [NEW] HyDE: 가상 문서(Hypothetical Document) 생성 함수
# ---------------------------------------------------------
def generate_hypothetical_document(query, model="gpt-4o-mini"):
    """
    질문에 대한 가상의 정답을 생성하여 검색용 확장 쿼리로 사용합니다.
    """
    system_prompt = """
    당신은 해당 분야의 전문가입니다. 사용자의 질문에 대해 핵심적인 키워드와 사실 위주로 간결하게 답변을 작성하세요.
    이 답변은 검색 엔진의 유사도 검색을 위한 '가상 문서(문맥 확장)'로 사용됩니다. 
    인사말, 서론, 결론을 절대 쓰지 말고 오직 정보와 키워드만 나열하세요.
    """
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": query}
            ],
            temperature=0.3 # 살짝의 창의성을 허용해 다양한 키워드를 뽑아냅니다.
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"HyDE API Error: {e}")
        return "" # 에러 시 빈 문자열 반환 (원본 쿼리만 쓰도록)

## RAG PipeLine Integration & Save

In [36]:
# ---------------------------------------------------------
# 1. 단일 샘플 처리 파이프라인 함수
# ---------------------------------------------------------
def process_single_sample(sample, top_k_hybrid=30, top_k_rerank=3):
    """
    eval.jsonl의 샘플 1개를 받아 전체 RAG 파이프라인을 거친 후,
    대회 제출 규격에 맞는 딕셔너리를 반환합니다.
    """
    eval_id = sample["eval_id"]
    msg_list = sample["msg"]
    
    # 1. Router & Rewriter (의도 파악 및 질의 재작성)
    routed_info = analyze_and_rewrite_query(msg_list)
    is_science = routed_info.get("is_science", False)
    standalone_query = routed_info.get("standalone_query", "")
    
    # 일상 대화(Chitchat)인 경우: 검색 없이 바로 LLM에게 일상 답변 지시
    if not is_science or not standalone_query:
        # 마지막 유저 메시지만 추출하여 일상 대화 생성
        last_user_msg = msg_list[-1]["content"] if msg_list else "안녕하세요!"

        # 문서가 없다고 에러를 뱉는 generate_final_answer 를 거치지 않고, 일상 대화 전용으로 LLM 을 직접 호출해보기
        try:
            chat_response = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": "당신은 다정하고 친근한 AI 조수입니다. 사용자의 일상적인 말에 구어체로 짧고 따뜻하게 공감하며 대답해주세요."},
                    {"role": "user", "content": last_user_msg}
                ],
                temperature=0.7
            )
            answer = chat_response.choices[0].message.content.strip()
        except:
            answer = "저도 그렇게 생각해요. 더 궁금한 내용이 있으신가요?"

        return {
            "eval_id": eval_id,
            "standalone_query": "[]", # 대회 베이스라인 규격 (검색 안함 표시)
            "topk": [],
            "answer": answer,
            "references": []
        }
    
    # ---------------------------------
    # 2. HyDE 적용 및 검색
    # ---------------------------------
    # 가상 문서 생성
    hypo_doc = generate_hypothetical_document(standalone_query)
    
    # 검색엔진에는 "원본 질문 + 가상 문서"를 던져서 풍부한 키워드로 검색!
    search_query = standalone_query + "\n" + hypo_doc
    hybrid_results = hybrid_search(search_query, k=top_k_hybrid)

    # ---------------------------------------------------------
    # 🌟 [유지] 3. 리랭킹 및 생성 (원본 질의만 사용!)
    # ---------------------------------------------------------
    # 리랭커는 팩트 체크를 해야 하므로 엄격하게 '원본 질문'으로만 평가합니다.
    reranked_results = rerank_hybrid_results(standalone_query, hybrid_results, top_k=top_k_rerank)
    
    # 🌟 0.9144점을 달성했던 [자가 검증]이 추가된 gpt-4o-mini Generator 사용
    answer = generate_final_answer(standalone_query, reranked_results, model="gpt-4o-mini")
    
    return {
        "eval_id": eval_id,
        "standalone_query": standalone_query,
        "topk": [res["docid"] for res in reranked_results],
        "answer": answer,
        "references": [{"score": res["rerank_score"], "content": res["content"]} for res in reranked_results]
    }

In [37]:
# ---------------------------------------------------------
# 2. 전체 평가 데이터(220개) 실행 루프
# ---------------------------------------------------------
submission_rows = []

for sample in tqdm(eval_data, desc="Processing Evaluation Data"):
    result_row = process_single_sample(sample, top_k_hybrid=30, top_k_rerank=3)
    submission_rows.append(result_row)

print("모든 Data 처리가 완료 되었습니다.")

Processing Evaluation Data:   1%|          | 2/220 [00:18<31:31,  8.68s/it]

Router API Error: Error code: 400 - {'error': {'message': "We could not parse the JSON body of your request. (HINT: This likely means you aren't using your HTTP library correctly. The OpenAI API expects a JSON payload, but what was sent was not valid JSON. If you have trouble figuring out how to fix this, please contact us through our help center at help.openai.com.)", 'type': 'invalid_request_error', 'param': None, 'code': None}}


Processing Evaluation Data: 100%|██████████| 220/220 [23:11<00:00,  6.33s/it]

모든 Data 처리가 완료 되었습니다.


In [38]:
# ---------------------------------------------------------
# 3. 결과 확인 및 저장
# ---------------------------------------------------------
submission_df = pd.DataFrame(submission_rows)

# 결과물 미리보기
display(submission_df.head(3))

# 저장 경로
# 2026.03.27 기준 submission_v1.jsonl -> SOTA 상태
save_csv_path = "../data/submission/submission_v6.csv"
save_jsonl_path = "../data/submission/submission_v6.jsonl"

# CSV 저장
submission_df.to_csv(save_csv_path, index=False, encoding="utf-8-sig")

# JSONL 저장
with open(save_jsonl_path, "w", encoding="utf-8") as f:
    for row in submission_rows:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(f"\n저장완료 \nCSV: {save_csv_path} \nJSONL: {save_jsonl_path}")

,eval_id,standalone_query,topk,answer,references
0,78,나무의 분류에 대해 조사하는 방법은 무엇인가요?,"[c63b9e3a-716f-423a-9c9b-0bcaa1b9f35d, 6788c97...",나무의 분류를 조사하는 방법은 여러 가지가 있어요. 한 가지 방법은 나무의 성장 속...,"[{'score': 0.9522578716278076, 'content': '한 학..."
1,213,각 나라에서의 공교육 지출 현황은 어떻게 되나요?,"[79c93deb-fe60-4c81-8d51-cb7400a0a156, 8843918...","각 나라의 공교육 지출 현황에 대한 구체적인 수치는 문서에 나와 있지 않지만, 전세...","[{'score': 0.9699116945266724, 'content': '201..."
2,107,[],[],어떤 원인인지 좀 더 구체적으로 말해줄래? 그러면 더 잘 도와줄 수 있을 것 같아!,[]



저장완료 
CSV: ../data/submission/submission_v6.csv 
JSONL: ../data/submission/submission_v6.jsonl


## 분석 

In [39]:
def run_local_diagnostics(submission_rows):
    '''
    제출용 데이터를 분석하여 로컬 평가 지표와 '취약 샘플' 을 추출해보기
    '''
    total_samples = len(submission_rows)
    searched_samples = 0
    skipped_samples = 0

    top1_scores = []
    score_gaps = []
    weak_cases = []     # 모델이 헷갈려하는 취약 샘플들

    for row in submission_rows:
        if row["standalone_query"] == "[]":
            skipped_samples += 1
            continue

        searched_samples += 1
        refs = row.get("references", [])

        if len(refs) >= 1:
            top1_score = refs[0]["score"]
            top1_scores.append(top1_score)

            # 1위와 2위의 점수 차이 (Gap이 클수록 1위 정답에 대한 확신이 큰 거임)
            if len(refs) >= 2:
                gap = top1_score - refs[1]["score"]
                score_gaps.append(gap)
            else:
                gap = 0.0

            # 취약 케이스 조건: 1위 점수가 너무 낮거나, 1/2위 점수 차이가 거의 없는 경우
            # Reranker 점수가 0에 가깝거나 음수면 관련성이 매우 낮다는 뜻
            if top1_score < 0.001 or gap < 0.0001:
                weak_cases.append({
                    "eval_id": row["eval_id"],
                    "query": row["standalone_query"],
                    "top1_score": top1_score,
                    "gap": gap,
                    "top1_doc": refs[0]["content"][:100] + "..."    # 내용 미리보기
                })
    
    # 전체 요약 통계 계산
    diagnostics_summary = {
        "전체 샘플 수": total_samples,
        "검색 수행 (과학)": searched_samples,
        "검색 스킵 (일상)": skipped_samples,
        "평균 Top-1 리랭커 점수": np.mean(top1_scores) if top1_scores else 0,
        "평균 1-2위 점수 격차 (Gap)": np.mean(score_gaps) if score_gaps else 0,
        "취약 샘플 (불확실성 높음)": len(weak_cases)
    }

    return diagnostics_summary, weak_cases

In [40]:
# ---------------------------------------------------------
# 진단 실행 및 결과 출력
# ---------------------------------------------------------
summary, weak_cases = run_local_diagnostics(submission_rows)

print("=== [로컬 진단 요약 리포트] ===")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"- {k}: {v:.6f}")
    else:
        print(f"- {k}: {v}")

print("\n=== [모델이 가장 헷갈려하는 취약 샘플 Top 5] ===")
# 취약 샘플 중에서도 점수가 가장 낮은 순으로 정렬해서 5개만 일단 보기
weak_cases_sorted = sorted(weak_cases, key=lambda x: x["top1_score"])[:5]

for idx, case in enumerate(weak_cases_sorted):
    print(f"\n[{idx+1}] eval_id: {case['eval_id']}")
    print(f"질문: {case['query']}")
    print(f"Top-1 점수: {case['top1_score']:.6f} | Gap: {case['gap']:.6f}")
    print(f"찾은 문서: {case['top1_doc']}")

=== [로컬 진단 요약 리포트] ===
- 전체 샘플 수: 220
- 검색 수행 (과학): 199
- 검색 스킵 (일상): 21
- 평균 Top-1 리랭커 점수: 0.807215
- 평균 1-2위 점수 격차 (Gap): 0.415420
- 취약 샘플 (불확실성 높음): 3

=== [모델이 가장 헷갈려하는 취약 샘플 Top 5] ===

[1] eval_id: 65
질문: 식초와 베이킹 소다를 섞으면 어떤 일이 일어나는가?
Top-1 점수: 0.999898 | Gap: 0.000091
찾은 문서: 식초와 베이킹소다를 혼합하면, 화학 반응이 일어나 소량의 기체가 생성됩니다. 이 기체는 혼합 용기 안에 갇혀 있을 것입니다. 따라서, 최종 생성물의 질량은 원래 식초와 베이킹소다의...

[2] eval_id: 53
질문: 하늘이 파란 이유는 무엇인가요?
Top-1 점수: 0.999933 | Gap: 0.000064
찾은 문서: 하늘이 파란 이유는 분자가 빨간빛보다 파란빛을 더 효과적으로 흩뜨리기 때문입니다. 파란색은 빛의 파장이 짧아서 빨간색보다 더 많은 빛을 흡수하고 반사합니다. 이로 인해 우리가 보는...

[3] eval_id: 43
질문: 달이 항상 같은 면만 보이는 이유는 무엇인가요?
Top-1 점수: 0.999953 | Gap: 0.000053
찾은 문서: 달의 본질적으로 같은 면을 항상 보는 이유는 달의 자전 주기와 궤도 주기가 같기 때문입니다. 달은 지구 주위를 공전하면서 동시에 자전을 합니다. 이때 달의 자전 주기와 궤도 주기가...


In [41]:
# ---------------------------------------------------------
# 라우터가 '일상 대화'로 분류하여 검색을 스킵한 케이스 분석
# ---------------------------------------------------------
skipped_cases = []

# submission_rows에서 스킵된(standalone_query가 "[]"인) 데이터의 eval_id를 찾고, 
# eval_data에서 해당 원본 질문을 매칭해서 가져옵니다.
for row in submission_rows:
    if row["standalone_query"] == "[]":
        target_id = row["eval_id"]
        
        # 원본 데이터에서 매칭되는 샘플 찾기
        original_sample = next((item for item in eval_data if item["eval_id"] == target_id), None)
        
        if original_sample:
            # 대화 히스토리 중 마지막 사용자의 발화(질문) 추출
            msg_list = original_sample["msg"]
            last_user_msg = ""
            for turn in reversed(msg_list):
                if turn.get("role") == "user":
                    last_user_msg = turn.get("content", "")
                    break
            
            skipped_cases.append({
                "eval_id": target_id,
                "original_query": last_user_msg,
                "generated_answer": row["answer"]
            })

print(f"==== 라우터가 스킵한 대화 목록 (총 {len(skipped_cases)}개) ====\n")

for idx, case in enumerate(skipped_cases):
    print(f"[{idx+1}] eval_id: {case['eval_id']}")
    print(f"👤 사용자 질문: {case['original_query']}")
    # 답변이 길 수 있으니 앞부분만 살짝 잘라서 보여줍니다.
    print(f"🤖 LLM의 일상 답변: {case['generated_answer'][:60]}...\n")
    print("-" * 50)

==== 라우터가 스킵한 대화 목록 (총 21개) ====

[1] eval_id: 107
👤 사용자 질문: 어떤 원인 때문에 발생하는지 궁금해.
🤖 LLM의 일상 답변: 어떤 원인인지 좀 더 구체적으로 말해줄래? 그러면 더 잘 도와줄 수 있을 것 같아!...

--------------------------------------------------
[2] eval_id: 276
👤 사용자 질문: 요새 너무 힘들다.
🤖 LLM의 일상 답변: 아, 그런 기분 이해해. 요즘 힘든 일이 많구나. 조금이라도 나아지길 바랄게. 필요한 거 있으면 언제든지 말...

--------------------------------------------------
[3] eval_id: 261
👤 사용자 질문: 니가 대답을 잘해줘서 너무 신나!
🤖 LLM의 일상 답변: 아, 정말 고마워! 그렇게 말해주니까 나도 너무 기뻐! 궁금한 거 있으면 언제든지 물어봐~ 😊...

--------------------------------------------------
[4] eval_id: 283
👤 사용자 질문: 이제 그만 얘기해!
🤖 LLM의 일상 답변: 알겠어! 필요할 때 언제든지 말해줘. 😊...

--------------------------------------------------
[5] eval_id: 32
👤 사용자 질문: 오늘 너무 즐거웠다!
🤖 LLM의 일상 답변: 와, 정말 좋다! 무슨 재미있는 일 있었어? 😊...

--------------------------------------------------
[6] eval_id: 94
👤 사용자 질문: 우울한데 신나는 얘기 좀 해줘!
🤖 LLM의 일상 답변: 아, 우울한 기분 이해해! 그럴 땐 신나는 이야기 듣는 게 정말 좋지! 예를 들어, 어떤 동물들이 서로 친구...

--------------------------------------------------
[7] eval_id: 90
👤 사용자 질문: 안녕

In [42]:
# ---------------------------------------------------------
# 라우터가 '과학/정보성 질문'으로 판단하여 검색을 수행한 케이스 점검
# ---------------------------------------------------------

searched_cases = []

for row in submission_rows:
    if row["standalone_query"] != "[]":
        target_id = row["eval_id"]
        
        # 원본 데이터에서 매칭되는 샘플 찾기
        original_sample = next((item for item in eval_data if item["eval_id"] == target_id), None)
        
        if original_sample:
            # 대화 히스토리 중 마지막 사용자의 발화(질문) 추출
            msg_list = original_sample["msg"]
            last_user_msg = ""
            for turn in reversed(msg_list):
                if turn.get("role") == "user":
                    last_user_msg = turn.get("content", "")
                    break
            
            searched_cases.append({
                "eval_id": target_id,
                "original_query": last_user_msg,
                "standalone_query": row["standalone_query"]
            })

print(f"==== 라우터가 검색을 수행한 목록 (총 {len(searched_cases)}개) ====\n")

# 너무 길면 보기 힘드니, 일단 무작위가 아닌 순서대로 50개씩 끊어서 보거나 전체를 출력합니다.
for idx, case in enumerate(searched_cases):
    print(f"[{idx+1}] eval_id: {case['eval_id']}")
    print(f"👤 원본 질문: {case['original_query']}")
    print(f"🔍 재생성된 질의: {case['standalone_query']}")
    print("-" * 50)

==== 라우터가 검색을 수행한 목록 (총 199개) ====

[1] eval_id: 78
👤 원본 질문: 나무의 분류에 대해 조사해 보기 위한 방법은?
🔍 재생성된 질의: 나무의 분류에 대해 조사하는 방법은 무엇인가요?
--------------------------------------------------
[2] eval_id: 213
👤 원본 질문: 각 나라에서의 공교육 지출 현황에 대해 알려줘.
🔍 재생성된 질의: 각 나라에서의 공교육 지출 현황은 어떻게 되나요?
--------------------------------------------------
[3] eval_id: 81
👤 원본 질문: 통학 버스의 가치에 대해 말해줘.
🔍 재생성된 질의: 통학 버스의 가치는 무엇인가요?
--------------------------------------------------
[4] eval_id: 280
👤 원본 질문: Dmitri Ivanovsky가 누구야?
🔍 재생성된 질의: Dmitri Ivanovsky가 누구인지 설명해 주세요.
--------------------------------------------------
[5] eval_id: 10
👤 원본 질문: 피임을 하기 위한 방법중 약으로 처리하는 방법은 쓸만한가?
🔍 재생성된 질의: 피임을 위한 약물 방법은 효과적인가?
--------------------------------------------------
[6] eval_id: 100
👤 원본 질문: 헬륨이 다른 원소들과 반응을 잘 안하는 이유는?
🔍 재생성된 질의: 헬륨이 다른 원소들과 반응을 잘 안하는 이유는 무엇인가요?
--------------------------------------------------
[7] eval_id: 279
👤 원본 질문: 문맹 비율이 사회 발전에 미치는 영향은?
🔍 재생성된 질의: 문맹 비율이 사회 발전에 미치는 영향은 무엇인가?
-------------------------------------

In [43]:
# 에러가 발생한 eval_id 찾기
error_ids = submission_df[submission_df['answer'] == "답변 생성 중 오류가 발생했어요."]['eval_id'].tolist()
print("🚨 에러 발생 eval_id:", error_ids)

🚨 에러 발생 eval_id: []
